# Barekatrezaei et al. (ESWA 2025): Laplace-Beltrami spectra + dual-path CNN

Our own reproduction of the first paper, from `reproduce_eswa2025.py` (written from the paper text).
Protocol: 5-fold CV on Train+Validation (130), stacking meta-learner, report on the official Test split (70).
Positive class = malignant. The authors' original code will be added at the end for comparison.

## Paper results (Table 3)

In [1]:
import pandas as pd
paper = pd.DataFrame({
    "AUC": [0.8233, 0.7771, 0.7867, 0.8725, 0.9350],
    "Acc": [0.7429, 0.6571, 0.6857, 0.7286, 0.8429],
    "Prec": [0.7037, 0.6667, 0.6552, 0.6842, 0.7959],
    "Sens": [0.9500, 0.8000, 0.9500, 0.9750, 0.9750],
    "Spec": [0.4667, 0.4667, 0.3333, 0.4000, 0.6667],
    "F1": [0.8085, 0.7273, 0.7755, 0.8041, 0.8764],
}, index=["SVM", "ExtraTrees", "CNN", "Voting", "Stacking"])
paper

,AUC,Acc,Prec,Sens,Spec,F1
SVM,0.8233,0.7429,0.7037,0.950,0.4667,0.8085
ExtraTrees,0.7771,0.6571,0.6667,0.800,0.4667,0.7273
CNN,0.7867,0.6857,0.6552,0.950,0.3333,0.7755
Voting,0.8725,0.7286,0.6842,0.975,0.4000,0.8041
Stacking,0.9350,0.8429,0.7959,0.975,0.6667,0.8764


## Features: radiomic, Laplace-Beltrami eigenvalues, 128x64x80 patches (cached)

In [2]:
import sys; sys.path.insert(0, "..")
import reproduce_eswa2025 as r
# r.build_cache() takes about 1.7 h; the Kaggle runs below already did it (see ../kaggle/README.md)

## Our results (Kaggle GPU runs, stored in `../results/eswa2025/`)

Seed 0 comes from the first run (gradient-boosting stacker only); seeds 1 and 2 also have a logistic-regression stacker. Re-run with `r.main(epochs=60, seed=s)` (about 1 h on a GPU).

In [3]:
from pathlib import Path
R = Path("../results/eswa2025")
runs = {f"seed {s}": pd.read_csv(R / f, index_col=0).rename(index={"Stacking": "Stacking-HGB"}) for s, f in [(0, "table3_seed0.csv"), (1, "results_seed1.csv"), (2, "results_seed2.csv")]}
ours = runs["seed 0"]
pd.DataFrame({k: v["AUC"] for k, v in runs.items()}).round(4)  # AUC per seed

,seed 0,seed 1,seed 2
CNN,0.8600,0.8283,0.8708
ExtraTrees,0.8125,0.8108,0.8088
SVM,0.7667,0.7767,0.7767
Stacking-HGB,0.7546,0.6787,0.7383
Stacking-LR,NaN,0.8242,0.8225
Voting,0.8625,0.8517,0.8633


## Ours minus paper

In [4]:
(ours.rename(index={"Stacking-HGB": "Stacking"}) - paper).round(4)

,AUC,Acc,Prec,Sens,Spec,F1
SVM,-0.0566,-0.0429,0.0074,-0.150,0.1000,-0.0556
ExtraTrees,0.0354,0.0286,0.0290,0.000,0.0666,0.0169
CNN,0.0733,0.0286,0.0448,-0.075,0.1667,0.0023
Voting,-0.0100,0.0428,0.0885,-0.125,0.2667,0.0054
Stacking,-0.1804,-0.2143,-0.1292,-0.275,-0.1334,-0.1935


## Ablations (paper Tables 4-8), seed 0\n\nRows are `table config [aggregator]`, aggregator = Voting, Stacking-HGB or Stacking-LR. The paper's Isomap/2-dim choice and the importance of the Laplace-Beltrami features do **not** reproduce.

In [5]:
abl = pd.read_csv(R / "ablations.csv", index_col=0)
abl

,AUC,Acc,Prec,Sens,Spec,F1
T4 isomap [Voting],0.8500,0.7571,0.7347,0.900,0.5667,0.8090
T4 isomap [Stacking-HGB],0.7800,0.6714,0.7429,0.650,0.7000,0.6933
T4 isomap [Stacking-LR],0.8142,0.7000,0.6939,0.850,0.5000,0.7640
T4 pca [Voting],0.8450,0.7714,0.7500,0.900,0.6000,0.8182
T4 pca [Stacking-HGB],0.7375,0.6857,0.7143,0.750,0.6000,0.7317
T4 pca [Stacking-LR],0.8208,0.7000,0.6939,0.850,0.5000,0.7640
T4 kpca [Voting],0.8450,0.7429,0.7391,0.850,0.6000,0.7907
T4 kpca [Stacking-HGB],0.8246,0.7857,0.8205,0.800,0.7667,0.8101
T4 kpca [Stacking-LR],0.8417,0.7286,0.7333,0.825,0.6000,0.7765
T4 lle [Voting],0.8567,0.7429,0.7292,0.875,0.5667,0.7955


## What we found about the Laplace-Beltrami features

5-fold CV on the 130 Train+Validation volumes, logistic regression, mean of 5 fold splits:

| eigenvalue version | corr. with lesion volume | LB only | + radiomic |
|---|---|---|---|
| raw log (ours) | -0.98 | 0.725 | 0.823 |
| area-normalised log(lambda*A) | -0.31 | 0.671 | 0.755 |
| ratio log(lambda_k/lambda_1) | 0.16 | 0.501 | 0.774 |
| radiomic only | | | 0.761 |

Raw eigenvalues help mostly as a lesion-size proxy; normalising makes them worse. The cause of the gap to the paper (stacking AUC 0.935 vs our 0.68-0.82) is still unknown.

## Authors' original code\n\nTODO: paste/import the code sent by the first author and compare against the tables above (start with the Laplace-Beltrami computation).

## Challenge benchmark

The TDSC-ABUS 2023 evaluation (`abus_classification/challenge.py`, explained in `challenge-benchmark.ipynb`), placed on
the published leaderboard, for each aggregator averaged over the seeds that ran it. **Classification only:** this method
classifies, and its features use the ground-truth masks, which challenge teams did not have.

In [6]:
from abus_classification import challenge
rows = pd.concat(runs.values()).rename(index={"Stacking": "Stacking-HGB"}).groupby(level=0)[["Acc", "AUC"]].mean()
challenge.benchmark({k: {"ACC": r.Acc, "AUC": r.AUC} for k, r in rows.iterrows()})[["ACC", "AUC", "Cls_Score", "Cls_Rank"]].round(3)

,ACC,AUC,Cls_Score,Cls_Rank
CNN,0.729,0.853,0.877,2/9
ExtraTrees,0.652,0.811,0.634,5/9
SVM,0.700,0.773,0.687,5/9
Stacking-HGB,0.614,0.724,0.410,5/9
Stacking-LR,0.693,0.823,0.748,4/9
Voting,0.752,0.859,0.942,2/9
